# Piping a RunnableParallel with Other Runnables

In [1]:
%load_ext dotenv
%dotenv

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

In [3]:
chat_template_books = ChatPromptTemplate.from_template(
    '''
    Suggest three of the best intermediate-level {programming language} books. 
    Answer only by listing the books.
    '''
)

chat_template_projects = ChatPromptTemplate.from_template(
    '''
    Suggest three interesting {programming language} projects suitable for intermediate-level programmers. 
    Answer only by listing the projects.
    '''
)

chat_template_time = ChatPromptTemplate.from_template(
     '''
     I'm an intermediate level programmer.
     
     Consider the following literature:
     {books}
     
     Also, consider the following projects:
     {projects}
     
     Roughly how much time would it take me to complete the literature and the projects?
     
     '''
)

In [4]:
chat = ChatGroq(model_name = "llama-3.1-8b-instant", 
                  model_kwargs = {'seed':365},
                  temperature = 0,
                  max_tokens = 500)

In [5]:
string_parser = StrOutputParser()

In [6]:
chain_books = chat_template_books | chat | string_parser

chain_projects = chat_template_projects | chat | string_parser

In [7]:
chain_parallel = RunnableParallel({'books':chain_books, 'projects':chain_projects})

In [8]:
chain_parallel.invoke({'programming language':'Python'})

{'books': '1. "Automate the Boring Stuff with Python" by Al Sweigart\n2. "Python Crash Course" by Eric Matthes\n3. "Learning Python" by Mark Lutz',
 'projects': '1. Web Scraper with Database Integration\n2. Chatbot using Natural Language Processing (NLP)\n3. Game Development with Pygame or Pyglet'}

In [9]:
chain_time1 = (RunnableParallel({'books':chain_books, 
                                'projects':chain_projects}) 
              | chat_template_time 
              | chat 
              | string_parser
             )

In [10]:
chain_time2 = ({'books':chain_books, 
                'projects':chain_projects}
              | chat_template_time 
              | chat 
              | string_parser
             )

In [11]:
print(chain_time2.invoke({'programming language':'Python'}))

To estimate the time required to complete the literature and projects, let's break it down into smaller tasks.

**Literature:**

1. "Automate the Boring Stuff with Python" by Al Sweigart:
   - Estimated reading time: 10-15 hours (depending on your reading speed and pace)
   - This book focuses on practical applications of Python, so it's a good resource for learning by doing.

2. "Python Crash Course" by Eric Matthes:
   - Estimated reading time: 20-25 hours (depending on your reading speed and pace)
   - This book covers a wide range of topics, including data structures, file input/output, and web development.

3. "Learning Python" by Mark Lutz:
   - Estimated reading time: 30-40 hours (depending on your reading speed and pace)
   - This book is a comprehensive resource that covers the language in-depth, including advanced topics like decorators and generators.

Assuming you dedicate 1-2 hours per day to reading, it would take you:

- 10-15 hours / 1 hour per day = 10-15 days to compl

In [12]:
chain_time2.get_graph().print_ascii()

            +-------------------------------+            
            | Parallel<books,projects>Input |            
            +-------------------------------+            
                   **               **                   
                ***                   ***                
              **                         **              
+--------------------+            +--------------------+ 
| ChatPromptTemplate |            | ChatPromptTemplate | 
+--------------------+            +--------------------+ 
           *                                 *           
           *                                 *           
           *                                 *           
     +----------+                      +----------+      
     | ChatGroq |                      | ChatGroq |      
     +----------+                      +----------+      
           *                                 *           
           *                                 *           
           *  